# 统一 28 天模型与滚动回测

所有模型只接收 `ForecastInputTask.history`，任务对象不含真实目标。预测输出同时保留 raw 和非负裁剪值；评价由 `track1/code/evaluate_predictions.py` 在预测完成后进行。

In [ ]:
from pathlib import Path
from datetime import date
import sys
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    def display(value): print(value.to_string() if hasattr(value, 'to_string') else value)
PROJECT = Path.cwd()
CODE = PROJECT / 'track1' / 'code'
assert CODE.is_dir(), '请将工作目录设为仓库根目录'
sys.path.insert(0, str(CODE))
from data_pipeline import load_all, make_input_task
from models import MODEL_NAMES, forecast
from run_backtest import run_one, run
print('Available models:', ', '.join(MODEL_NAMES))


## 模型来源与性质

Seasonal Naïve 是必备基线；ETS 为本次新增；SARIMA 受 P1 的 `Fighting BI Team` 方法启发。LightGBM 迁移自 `6104_CJB`；传统回归迁移自 `Machine Not Learning` / `Parrot Nothing`；Transformer 是为数值序列另写的本地小模型，不是预训练基础模型。

In [ ]:
families = pd.DataFrame([
    ('seasonal_naive','统计基线','最后一周重复'),
    ('ets','新增统计基线','7天加性季节'),
    ('sarima','Fighting BI Team 方法迁移','周季节统计模型'),
    ('lightgbm','6104_CJB 迁移','每步长一模型'),
    ('random_forest','Machine Not Learning 迁移','每步长一模型'),
    ('parrot_random_forest_800','Parrot Nothing 迁移','800树，每步长一模型'),
    ('ridge / linear_regression / gradient_boosting / svr / lasso / elasticnet','传统回归对照','每步长一模型'),
    ('transformer','数值序列新改写','56天输入，28天输出'),
], columns=['模型','来源','输入与输出方式'])
display(families)


## 单任务演示

可修改市场和起点。使用该起点的前 180 天历史，输出未来 28 天路径；不将未来实际值传入模型。默认演示 Seasonal Naïve、ETS、SARIMA，运行较快。把模型名添加到 `demo_models` 即可运行 LightGBM、Random Forest 或 Transformer。

In [ ]:
series = load_all()
task = make_input_task(series['Macau:CN_Chinese_mainland'], date(2026, 6, 14), history_length=180)
demo_models = ['seasonal_naive', 'ets', 'sarima']
forecast_rows, time_rows = [], []
for name in demo_models:
    rows, timing = run_one(task, name)
    forecast_rows.extend(rows)
    time_rows.append(timing)
predictions = pd.DataFrame(forecast_rows)
display(predictions[['model','target_date','horizon_day','prediction_raw','prediction','was_clipped']].head(12))
display(pd.DataFrame(time_rows)[['model','fit_seconds','inference_seconds','total_seconds']])
assert predictions.groupby('model').size().eq(28).all()
assert 'actual' not in predictions.columns


## 对齐论文的四个提前期

本阶段只查看预测。MASE、RMSSE、绝对误差和高需求日指标由独立评估器在合并 targets 后计算。

In [ ]:
key_horizons = predictions[predictions['horizon_day'].isin([1,7,14,28])]
display(key_horizons[['model','horizon_day','target_date','prediction_raw','prediction','was_clipped']])


## 完整滚动实验（默认关闭）

正式外层实验要先在 B 组完成调参并冻结配置；当前配置仍为 `pending_B_group_tuning`。以下开关默认关闭，且在参数未冻结时会拒绝批量运行。

In [ ]:
RUN_PAPER_BACKTEST = False
if RUN_PAPER_BACKTEST:
    from baseline_config import validate
    audit = validate(task_manifest_path=PROJECT / 'shared' / 'data' / 'task_manifest.json')
    if audit['freeze_status'] != 'frozen_after_B_group_tuning':
        raise RuntimeError('请先完成 B 组调参并冻结 baseline 配置')
    selected = ('seasonal_naive','ets','sarima','random_forest','lightgbm','transformer')
    for destination in ('Macau','HongKong'):
        result = run(inputs=PROJECT / 'shared' / 'data' / 'task_inputs.parquet',
                     model_names=selected, destinations=(destination,),
                     history_lengths=('all',), splits=('outer_backtest',),
                     output=PROJECT / 'track1' / 'results' / destination.lower())
        print(destination, result['forecast_tasks'], result['prediction_rows'])


**当前边界：**该文件夹尚未接入 TimesFM、Chronos、Moirai 或多智能体 API；也未生成节假日/演唱会标签或云实例费用。现有的小型 Transformer 应归为本地训练模型。